# Café Customer Analytics & Churn Prediction
### From transaction data to customer segmentation, churn prediction, and retention strategy

**Portfolio Project**

This project reconstructs and extends the skills learned in a live Python Data Analysis & AI course.  
Using a course-provided café transaction dataset, the project moves from descriptive analytics to customer segmentation and predictive modeling.

### Business Questions
1. How did revenue, order volume, and active customers change during 2025?
2. Which customer groups and product categories contribute most to performance?
3. Can customers be segmented using behavioral data?
4. Can customer churn be predicted before customers disappear?
5. How can churn probability and customer value be combined to prioritize retention?
6. What does the coupon pilot suggest about different incentive levels?

### Technical Skills Demonstrated
- Python, Pandas, NumPy
- Data cleaning and feature engineering
- Exploratory data analysis and visualization
- Pareto analysis
- RFM-style customer segmentation
- K-Means clustering and PCA
- Leakage-aware churn labeling
- Scikit-learn preprocessing pipelines
- Cross-validation and held-out testing
- Logistic Regression, Decision Tree, Random Forest
- ROC-AUC, Precision, Recall, F1 Score
- Permutation feature importance
- Customer risk prioritization
- Coupon pilot analysis

> **Project context:** This is a portfolio reconstruction based on techniques learned during the course. The analysis below is my own organized implementation using the course datasets.
> **Colab note:** Use **Runtime → Run all**. The notebook creates the monthly summary before plotting and saves the combined Revenue / Orders / Active Customers chart to `images/monthly_activity_trend.png`.

> **Image export:** Every main visualization in this notebook is automatically saved as a 300-dpi PNG to `MyDrive/Cafe_Customer_Analytics/images/` when you use **Runtime → Run all**.


## Table of Contents

1. Environment & Data Loading  
2. Data Audit & Cleaning  
3. Business Performance Overview  
4. Trend Decomposition  
5. Customer Concentration & Pareto Analysis  
6. Product, Channel, Store & Time Analysis  
7. Customer Behavioral Profile  
8. Customer Segmentation with K-Means  
9. Churn Definition & Feature Engineering  
10. Machine Learning with Cross-Validation  
11. Held-Out Test Evaluation  
12. Churn Driver Interpretation  
13. Customer Risk Prioritization  
14. Coupon Pilot Analysis  
15. Recommended Actions  
16. Limitations & Next Steps  
17. Export Portfolio Outputs

## 1. Environment & Data Loading

In [ ]:
# Core libraries
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    silhouette_score
)

from sklearn.inspection import permutation_importance

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

sns.set_context('notebook')

In [ ]:
# Mount Google Drive when running in Google Colab.
# If Drive is already mounted, Colab will simply reuse it.

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab.")

In [ ]:
# File locations
PROJECT_DIR = '/content/drive/MyDrive/Cafe_Customer_Analytics'
DATA_DIR = f'{PROJECT_DIR}/data'
OUTPUT_DIR = f'{PROJECT_DIR}/outputs'
IMAGE_DIR = f'{PROJECT_DIR}/images'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

ORDERS_PATH = f'{DATA_DIR}/cafe_orders.csv'
COUPON_PATH = f'{DATA_DIR}/coupon_pilot.csv'

# Load datasets
orders_raw = pd.read_csv(ORDERS_PATH)
coupon_raw = pd.read_csv(COUPON_PATH)

print("Orders dataset:", orders_raw.shape)
print("Coupon dataset:", coupon_raw.shape)

display(orders_raw.head())
display(coupon_raw.head())

## 2. Data Audit & Cleaning

Before analysis, the dataset is checked for:
- data types
- missing values
- duplicate records
- duplicate order IDs
- invalid transaction values
- date coverage

The original dataframe is preserved and a cleaned working copy is created.

In [ ]:
# Dataset structure
orders_raw.info()

In [ ]:
# Missing-value audit
missing_audit = pd.DataFrame({
    'missing_count': orders_raw.isna().sum(),
    'missing_pct': orders_raw.isna().mean() * 100
}).sort_values('missing_pct', ascending=False)

display(missing_audit)

In [ ]:
# Data-quality checks
print("Duplicate rows:", orders_raw.duplicated().sum())
print("Duplicate order IDs:", orders_raw['order_id'].duplicated().sum())
print("Non-positive transaction amounts:", (orders_raw['final_amount'] <= 0).sum())

print("\nAge range:")
display(orders_raw['age'].describe())

In [ ]:
# Create a clean working copy
orders = orders_raw.copy()

# Convert time
orders['order_received_at'] = pd.to_datetime(
    orders['order_received_at'],
    errors='coerce'
)

# Missing categorical values with meaningful labels
orders['gender'] = orders['gender'].fillna('Unknown')
orders['membership'] = orders['membership'].fillna('Non-member')
orders['discount_type'] = orders['discount_type'].fillna('No Discount')

# Age remains missing here.
# It will be handled inside the ML preprocessing pipeline to avoid leakage.

# Time features
orders['date'] = orders['order_received_at'].dt.date
orders['month'] = orders['order_received_at'].dt.to_period('M').astype(str)
orders['month_num'] = orders['order_received_at'].dt.month
orders['day_of_week'] = orders['order_received_at'].dt.day_name()
orders['hour'] = orders['order_received_at'].dt.hour

print("Date range:", orders['order_received_at'].min(), "to", orders['order_received_at'].max())
print("Orders:", f"{len(orders):,}")
print("Customers:", f"{orders['user_id'].nunique():,}")

display(orders.isna().sum())

### Cleaning Decision

`membership` and `discount_type` missing values are treated as meaningful categories rather than automatically replaced with the most common value.

`age` is intentionally **not** globally imputed.  
For predictive modeling, age imputation is performed only inside the training pipeline, preventing information from the test set from influencing preprocessing.

## 3. Business Performance Overview

In [ ]:
total_revenue = orders['final_amount'].sum()
total_orders = len(orders)
total_customers = orders['user_id'].nunique()
avg_order_value = orders['final_amount'].mean()
reorder_rate = orders['is_reorder'].mean()

overview = pd.DataFrame({
    'Metric': [
        'Total Revenue',
        'Total Orders',
        'Unique Customers',
        'Average Order Value',
        'Reorder Rate'
    ],
    'Value': [
        f"{total_revenue:,.0f}",
        f"{total_orders:,}",
        f"{total_customers:,}",
        f"{avg_order_value:,.0f}",
        f"{reorder_rate:.1%}"
    ]
})

display(overview)

## 4. Trend Decomposition

In [ ]:
monthly = orders.groupby('month').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count'),
    active_customers=('user_id', 'nunique')
).reset_index()

monthly['avg_order_value'] = monthly['revenue'] / monthly['orders']
monthly['orders_per_customer'] = monthly['orders'] / monthly['active_customers']

display(monthly)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(monthly['month'], monthly['revenue'] / 1_000_000, marker='o')
axes[0, 0].set_title('Monthly Revenue')
axes[0, 0].set_ylabel('Revenue (Million)')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].plot(monthly['month'], monthly['orders'], marker='o')
axes[0, 1].set_title('Monthly Order Volume')
axes[0, 1].set_ylabel('Orders')
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].plot(monthly['month'], monthly['active_customers'], marker='o')
axes[1, 0].set_title('Monthly Active Customers')
axes[1, 0].set_ylabel('Customers')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].plot(monthly['month'], monthly['avg_order_value'], marker='o')
axes[1, 1].set_title('Monthly Average Order Value')
axes[1, 1].set_ylabel('Average Order Value')
axes[1, 1].tick_params(axis='x', rotation=45)

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/monthly_decomposition.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

### Portfolio Figure: Revenue, Orders & Active Customers

This figure combines the three monthly activity measures used in the portfolio. Revenue uses the left axis because it is measured in monetary units, while orders and active customers use the right axis because they are counts. The chart is generated directly from the `monthly` dataframe created above, so the portfolio image is reproducible from the notebook.


In [ ]:
# Portfolio-ready combined trend chart
# This cell is intentionally self-contained with respect to plotting:
# `plt` was imported in Section 1 and `monthly` was created immediately above.

required_columns = {'month', 'revenue', 'orders', 'active_customers'}
missing_columns = required_columns - set(monthly.columns)

if missing_columns:
    raise ValueError(
        f"monthly is missing required columns: {sorted(missing_columns)}"
    )

fig, ax1 = plt.subplots(figsize=(12, 6))

# Revenue — left y-axis
line1 = ax1.plot(
    monthly['month'],
    monthly['revenue'] / 1_000_000,
    marker='o',
    linewidth=2,
    label='Revenue'
)

ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue (Million)')
ax1.set_title('Revenue decline tracks orders and active customers')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(alpha=0.25)

# Orders and active customers — right y-axis
ax2 = ax1.twinx()

line2 = ax2.plot(
    monthly['month'],
    monthly['orders'],
    marker='o',
    linewidth=2,
    label='Orders'
)

line3 = ax2.plot(
    monthly['month'],
    monthly['active_customers'],
    marker='o',
    linewidth=2,
    label='Active Customers'
)

ax2.set_ylabel('Count')

# One combined legend
lines = line1 + line2 + line3
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='upper right')

plt.tight_layout()

# Save the exact notebook-generated figure for the portfolio
portfolio_chart_path = f'{IMAGE_DIR}/monthly_activity_trend.png'
plt.savefig(portfolio_chart_path, dpi=300, bbox_inches='tight')

plt.show()

print(f"Saved portfolio chart to: {portfolio_chart_path}")


In [ ]:
# Quantify January-to-December change
jan = monthly.iloc[0]
dec = monthly.iloc[-1]

trend_change = pd.DataFrame({
    'Metric': ['Revenue', 'Orders', 'Active Customers', 'Average Order Value'],
    'Jan': [jan['revenue'], jan['orders'], jan['active_customers'], jan['avg_order_value']],
    'Dec': [dec['revenue'], dec['orders'], dec['active_customers'], dec['avg_order_value']]
})

trend_change['Change %'] = (
    (trend_change['Dec'] / trend_change['Jan']) - 1
) * 100

display(trend_change.round(2))

### Interpretation

The purpose of this decomposition is to avoid treating revenue as a single isolated number.

If revenue decreases while average order value stays relatively stable, the more likely operational issue is declining customer activity or order frequency rather than customers spending dramatically less per transaction.

## 5. Customer Concentration & Pareto Analysis

In [ ]:
customer_value = orders.groupby('user_id').agg(
    total_spending=('final_amount', 'sum'),
    total_orders=('order_id', 'count'),
    avg_order_value=('final_amount', 'mean')
).reset_index()

customer_value = customer_value.sort_values(
    'total_spending',
    ascending=False
).reset_index(drop=True)

customer_value['cumulative_revenue_pct'] = (
    customer_value['total_spending'].cumsum()
    / customer_value['total_spending'].sum()
    * 100
)

customer_value['cumulative_customer_pct'] = (
    (customer_value.index + 1)
    / len(customer_value)
    * 100
)

top_20_n = int(np.ceil(len(customer_value) * 0.20))
top_20_share = (
    customer_value.head(top_20_n)['total_spending'].sum()
    / customer_value['total_spending'].sum()
    * 100
)

print(f"Top 20% of customers generate {top_20_share:.1f}% of total revenue.")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    customer_value['cumulative_customer_pct'],
    customer_value['cumulative_revenue_pct'],
    linewidth=2
)

plt.axvline(20, linestyle='--', alpha=0.7)
plt.axhline(top_20_share, linestyle='--', alpha=0.7)

plt.scatter([20], [top_20_share], s=70)
plt.annotate(
    f'Top 20% → {top_20_share:.1f}% of revenue',
    (20, top_20_share),
    xytext=(28, max(10, top_20_share - 15)),
    arrowprops=dict(arrowstyle='->')
)

plt.title('Customer Revenue Concentration (Pareto Curve)')
plt.xlabel('Cumulative Customers (%)')
plt.ylabel('Cumulative Revenue (%)')
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.grid(alpha=0.25)
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/pareto_curve.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
# Behavioral customer tiers based on order frequency
customer_behavior = orders.groupby('user_id').agg(
    visits=('order_id', 'count'),
    total_spend=('final_amount', 'sum'),
    avg_ticket=('final_amount', 'mean')
).reset_index()

customer_behavior['frequency_tier'] = pd.qcut(
    customer_behavior['visits'].rank(method='first'),
    q=[0, 0.5, 0.8, 1],
    labels=['Light', 'Medium', 'Heavy']
)

frequency_summary = customer_behavior.groupby(
    'frequency_tier',
    observed=False
).agg(
    customers=('user_id', 'count'),
    avg_visits=('visits', 'mean'),
    avg_ticket=('avg_ticket', 'mean'),
    total_revenue=('total_spend', 'sum')
).reset_index()

frequency_summary['revenue_share_pct'] = (
    frequency_summary['total_revenue']
    / frequency_summary['total_revenue'].sum()
    * 100
)

display(frequency_summary.round(2))

## 6. Product, Channel, Store & Time Analysis

In [ ]:
category_summary = orders.groupby('category').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count'),
    customers=('user_id', 'nunique')
).sort_values('revenue', ascending=False)

category_summary['revenue_share_pct'] = (
    category_summary['revenue'] / category_summary['revenue'].sum() * 100
)

display(category_summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

category_plot = category_summary.reset_index()

sns.barplot(
    data=category_plot,
    x='category',
    y='revenue_share_pct',
    ax=axes[0]
)
axes[0].set_title('Revenue Share by Product Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Revenue Share (%)')
axes[0].tick_params(axis='x', rotation=30)

category_month = orders.groupby(
    ['month', 'category']
)['order_id'].count().reset_index(name='orders')

sns.lineplot(
    data=category_month,
    x='month',
    y='orders',
    hue='category',
    marker='o',
    ax=axes[1]
)
axes[1].set_title('Monthly Orders by Product Category')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Orders')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/category_analysis.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
# Category growth from January to December
category_pivot = orders.groupby(
    ['month', 'category']
)['order_id'].count().unstack(fill_value=0)

category_growth = pd.DataFrame({
    'January Orders': category_pivot.iloc[0],
    'December Orders': category_pivot.iloc[-1]
})

category_growth['Jan-Dec Change %'] = (
    category_growth['December Orders']
    / category_growth['January Orders']
    - 1
) * 100

display(category_growth.sort_values('Jan-Dec Change %', ascending=False).round(2))

In [ ]:
order_source_summary = orders.groupby('order_source').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count'),
    avg_order_value=('final_amount', 'mean')
).sort_values('revenue', ascending=False)

store_summary = orders.groupby('store_type').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count'),
    customers=('user_id', 'nunique'),
    avg_order_value=('final_amount', 'mean')
).sort_values('revenue', ascending=False)

print("Order Source")
display(order_source_summary.round(2))

print("\nStore Type")
display(store_summary.round(2))

In [ ]:
day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

weekday_summary = orders.groupby('day_of_week').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count')
).reindex(day_order)

hourly_summary = orders.groupby('hour').agg(
    revenue=('final_amount', 'sum'),
    orders=('order_id', 'count')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    x=weekday_summary.index,
    y=weekday_summary['orders'],
    ax=axes[0]
)
axes[0].set_title('Orders by Day of Week')
axes[0].set_xlabel('')
axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=35)

axes[1].plot(hourly_summary.index, hourly_summary['orders'], marker='o')
axes[1].set_title('Orders by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Orders')
axes[1].grid(alpha=0.25)

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/time_analysis.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

## 7. Customer Behavioral Profile

In [ ]:
customer_profile = orders.groupby('user_id').agg(
    total_spending=('final_amount', 'sum'),
    total_orders=('order_id', 'count'),
    avg_order_value=('final_amount', 'mean'),
    first_order=('order_received_at', 'min'),
    last_order=('order_received_at', 'max'),
    reorder_rate=('is_reorder', 'mean'),
    category_diversity=('category', 'nunique')
).reset_index()

display(customer_profile.head())
display(customer_profile.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(
    customer_profile['total_spending'],
    bins=35,
    ax=axes[0]
)
axes[0].set_title('Distribution of Customer Total Spending')

sns.scatterplot(
    data=customer_profile,
    x='total_orders',
    y='total_spending',
    alpha=0.5,
    ax=axes[1]
)
axes[1].set_title('Order Frequency vs Total Spending')

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/customer_behavior_profile.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

## 8. Customer Segmentation with K-Means

An RFM-style behavioral feature set is constructed:

- **Recency** — days since the latest purchase
- **Frequency** — number of orders
- **Monetary** — total spending
- **Average Order Value**
- **Reorder Rate**

Frequency and Monetary are log-transformed before scaling to reduce skew.

The number of clusters is evaluated using both:
- inertia (Elbow Method)
- Silhouette Score

PCA is then used only for visualization; clustering itself is performed on the standardized behavioral features.

In [ ]:
reference_date = orders['order_received_at'].max() + pd.Timedelta(days=1)

rfm = orders.groupby('user_id').agg(
    Recency=('order_received_at', lambda x: (reference_date - x.max()).days),
    Frequency=('order_id', 'count'),
    Monetary=('final_amount', 'sum'),
    AvgOrderValue=('final_amount', 'mean'),
    ReorderRate=('is_reorder', 'mean')
).reset_index()

rfm_model = rfm[
    ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'ReorderRate']
].copy()

rfm_model['Frequency'] = np.log1p(rfm_model['Frequency'])
rfm_model['Monetary'] = np.log1p(rfm_model['Monetary'])

cluster_scaler = StandardScaler()
rfm_scaled = cluster_scaler.fit_transform(rfm_model)

display(rfm.head())

In [ ]:
cluster_diagnostics = []

for k in range(2, 7):
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )
    labels = km.fit_predict(rfm_scaled)

    cluster_diagnostics.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette_score': silhouette_score(rfm_scaled, labels)
    })

cluster_diagnostics = pd.DataFrame(cluster_diagnostics)
display(cluster_diagnostics.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    cluster_diagnostics['k'],
    cluster_diagnostics['inertia'],
    marker='o'
)
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')
axes[0].grid(alpha=0.25)

axes[1].plot(
    cluster_diagnostics['k'],
    cluster_diagnostics['silhouette_score'],
    marker='o'
)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(alpha=0.25)

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/cluster_diagnostics.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
best_k = int(
    cluster_diagnostics.loc[
        cluster_diagnostics['silhouette_score'].idxmax(),
        'k'
    ]
)

print("Best k by Silhouette Score:", best_k)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby('cluster').agg(
    customers=('user_id', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    avg_order_value=('AvgOrderValue', 'mean'),
    avg_reorder_rate=('ReorderRate', 'mean')
).round(2)

display(cluster_summary)

In [ ]:
# PCA visualization of the clusters
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)

rfm['PC1'] = rfm_pca[:, 0]
rfm['PC2'] = rfm_pca[:, 1]

print(
    "Variance explained by two PCA components:",
    f"{pca.explained_variance_ratio_.sum():.1%}"
)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=rfm,
    x='PC1',
    y='PC2',
    hue='cluster',
    palette='tab10',
    alpha=0.65,
    s=35
)

plt.title('Customer Segments Projected into Two PCA Dimensions')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/pca_segments.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

### Important Interpretation Note

Clustering is an **unsupervised** method.  
The cluster numbers themselves do not mean “VIP,” “normal,” or “at-risk.”

Business labels should only be assigned after examining the actual behavioral profile of each cluster.

## 9. Churn Definition & Feature Engineering

To avoid future-data leakage:

- **Observation period:** January–September 2025  
- **Outcome period:** October–December 2025  

A customer is labeled **churn = 1** if they purchased during the observation period but did not return during the outcome period.

Only information available by the end of September is used to create predictive features.

In [ ]:
observation_end = pd.Timestamp('2025-09-30 23:59:59')
outcome_start = pd.Timestamp('2025-10-01 00:00:00')

observation = orders[
    orders['order_received_at'] <= observation_end
].copy()

outcome = orders[
    orders['order_received_at'] >= outcome_start
].copy()

print("Observation-period orders:", f"{len(observation):,}")
print("Outcome-period orders:", f"{len(outcome):,}")

In [ ]:
def mode_or_nan(series):
    values = series.dropna().mode()
    return values.iloc[0] if len(values) else np.nan

customer_features = observation.groupby('user_id').agg(
    # Core behavioral variables
    recency=('order_received_at', lambda x: (observation_end - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('final_amount', 'sum'),
    avg_order_value=('final_amount', 'mean'),
    reorder_rate=('is_reorder', 'mean'),

    # Engagement / diversity variables
    active_months=('order_received_at', lambda x: x.dt.to_period('M').nunique()),
    category_diversity=('category', 'nunique'),
    mobile_share=('order_source', lambda x: (x == 'Mobile_App').mean()),
    delivery_share=('order_type', lambda x: (x == 'Delivery').mean()),
    coupon_share=('discount_type', lambda x: (x == 'Coupon').mean()),

    # Customer profile variables
    age=('age', 'median'),
    gender=('gender', mode_or_nan),
    membership=('membership', mode_or_nan),
    store_type=('store_type', mode_or_nan),
    order_source=('order_source', mode_or_nan),
    favorite_category=('category', mode_or_nan)
).reset_index()

returned_customers = set(outcome['user_id'].unique())

customer_features['churn'] = (
    ~customer_features['user_id'].isin(returned_customers)
).astype(int)

display(customer_features.head())

In [ ]:
churn_summary = customer_features['churn'].value_counts().rename_axis(
    'churn'
).reset_index(name='customers')

churn_summary['percentage'] = (
    churn_summary['customers']
    / churn_summary['customers'].sum()
    * 100
)

display(churn_summary)

print(
    "Overall churn rate:",
    f"{customer_features['churn'].mean():.1%}"
)

In [ ]:
plt.figure(figsize=(6, 4))

sns.barplot(
    data=churn_summary,
    x='churn',
    y='percentage'
)

plt.title('Customer Churn in Outcome Period')
plt.xlabel('Churn Label (0 = Returned, 1 = Churned)')
plt.ylabel('Customers (%)')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/churn_distribution.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

## 10. Machine Learning with Cross-Validation

Three models are compared:

1. Logistic Regression
2. Decision Tree
3. Random Forest

### Why use a preprocessing pipeline?

Numeric variables:
- median imputation
- standardization

Categorical variables:
- most-frequent imputation
- one-hot encoding

The pipeline ensures that preprocessing is learned only from the training data.

### Why use cross-validation?

A single train/test split can be lucky or unlucky.  
Five-fold stratified cross-validation provides a more stable comparison before final evaluation on a completely held-out test set.

In [ ]:
numeric_features = [
    'recency',
    'frequency',
    'monetary',
    'avg_order_value',
    'reorder_rate',
    'active_months',
    'category_diversity',
    'mobile_share',
    'delivery_share',
    'coupon_share',
    'age'
]

categorical_features = [
    'gender',
    'membership',
    'store_type',
    'order_source',
    'favorite_category'
]

feature_columns = numeric_features + categorical_features

X = customer_features[feature_columns]
y = customer_features['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Held-out test customers:", len(X_test))
print("Training churn rate:", f"{y_train.mean():.1%}")
print("Test churn rate:", f"{y_test.mean():.1%}")

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=20,
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'Accuracy': 'accuracy',
    'Precision': 'precision',
    'Recall': 'recall',
    'F1': 'f1',
    'ROC_AUC': 'roc_auc'
}

cv_rows = []
pipelines = {}

for model_name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipelines[model_name] = pipeline

    cv_result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    row = {'Model': model_name}

    for metric in scoring.keys():
        row[f'CV {metric}'] = cv_result[f'test_{metric}'].mean()
        row[f'CV {metric} SD'] = cv_result[f'test_{metric}'].std()

    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values(
    'CV ROC_AUC',
    ascending=False
)

display(cv_results.round(3))

In [ ]:
# Visualize the main cross-validation metrics
cv_plot = cv_results.set_index('Model')[
    ['CV Precision', 'CV Recall', 'CV F1', 'CV ROC_AUC']
]

cv_plot.plot(
    kind='bar',
    figsize=(11, 5)
)

plt.title('5-Fold Cross-Validation Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/model_cross_validation.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

### Model Selection Rule

The model with the highest mean **cross-validated ROC-AUC** is selected automatically.

ROC-AUC is used for model selection because this project is a ranking problem as well as a classification problem: the model should place genuinely high-risk customers above low-risk customers across many possible probability thresholds.

F1, Recall, and Precision are still reported because operational campaigns depend on the trade-off between:
- missing churners
- contacting too many false positives

In [ ]:
best_model_name = cv_results.iloc[0]['Model']
best_model = pipelines[best_model_name]

print("Selected model:", best_model_name)

best_model.fit(X_train, y_train)

## 11. Held-Out Test Evaluation

In [ ]:
test_rows = []

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]

    test_rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC_AUC': roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).sort_values(
    'ROC_AUC',
    ascending=False
)

display(test_results.round(3))

In [ ]:
# Final held-out predictions from the selected model
best_model.fit(X_train, y_train)

test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)[:, 1]

print("Selected model:", best_model_name)
print()
print(classification_report(y_test, test_pred))

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cbar=False
)

plt.title(f'{best_model_name}: Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/confusion_matrix.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(
    y_test,
    test_prob,
    ax=axes[0]
)
axes[0].set_title(f'{best_model_name}: ROC Curve')

PrecisionRecallDisplay.from_predictions(
    y_test,
    test_prob,
    ax=axes[1]
)
axes[1].set_title(f'{best_model_name}: Precision–Recall Curve')

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/roc_pr_curves.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

## 12. Churn Driver Interpretation

Permutation importance measures how much model performance decreases when one feature is randomly shuffled.

Unlike raw tree importance, this approach can be applied to the final pipeline using the original feature columns, making the result easier to explain.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=12,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': perm.importances_mean,
    'Std': perm.importances_std
}).sort_values('Importance', ascending=False)

display(importance.round(4))

In [ ]:
plt.figure(figsize=(9, 6))

sns.barplot(
    data=importance.head(12),
    x='Importance',
    y='Feature'
)

plt.title(f'Top Churn Predictors — {best_model_name}')
plt.xlabel('Decrease in ROC-AUC When Shuffled')
plt.ylabel('')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/feature_importance.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

## 13. Customer Risk Prioritization

A churn probability alone does not tell a business whom to contact first.

To create an operational priority score, the analysis combines:

- predicted churn probability
- a simple three-month revenue proxy based on each customer's observed monthly spending rate

This is **not** a forecast of guaranteed future revenue.  
It is a transparent prioritization heuristic designed to identify customers who are both:
1. likely to churn, and
2. financially meaningful.

In [ ]:
# Refit selected model on all observation-period customers
best_model.fit(X, y)

customer_features['churn_probability'] = (
    best_model.predict_proba(X)[:, 1]
)

# Historical monthly spending rate during observation period
customer_features['monthly_spend_proxy'] = (
    customer_features['monetary']
    / customer_features['active_months'].clip(lower=1)
)

# Three-month revenue proxy
customer_features['three_month_value_proxy'] = (
    customer_features['monthly_spend_proxy'] * 3
)

# Expected value at risk heuristic
customer_features['value_at_risk_score'] = (
    customer_features['churn_probability']
    * customer_features['three_month_value_proxy']
)

risk_ranking = customer_features.sort_values(
    'value_at_risk_score',
    ascending=False
)

display(
    risk_ranking[
        [
            'user_id',
            'churn_probability',
            'three_month_value_proxy',
            'value_at_risk_score',
            'recency',
            'frequency',
            'monetary',
            'membership'
        ]
    ].head(20).round(2)
)

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=customer_features,
    x='churn_probability',
    y='three_month_value_proxy',
    size='value_at_risk_score',
    sizes=(15, 180),
    alpha=0.45,
    legend=False
)

plt.title('Retention Priority Map')
plt.xlabel('Predicted Churn Probability')
plt.ylabel('3-Month Revenue Proxy')
plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/risk_map.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
# Example operational shortlist:
# customers in the top 10% by value-at-risk score

risk_cutoff = customer_features['value_at_risk_score'].quantile(0.90)

priority_customers = customer_features[
    customer_features['value_at_risk_score'] >= risk_cutoff
].sort_values(
    'value_at_risk_score',
    ascending=False
)

print("Priority customers:", len(priority_customers))
print(
    "Average churn probability:",
    f"{priority_customers['churn_probability'].mean():.1%}"
)

display(
    priority_customers[
        [
            'user_id',
            'churn_probability',
            'three_month_value_proxy',
            'value_at_risk_score'
        ]
    ].head(15).round(2)
)

## 14. Coupon Pilot Analysis

The second dataset contains:
- customer ID
- pilot group
- coupon amount
- 30-day campaign spending

The analysis compares mean customer spending across coupon groups.

> **Causal interpretation warning:** The dataset name suggests a pilot, but random assignment is not independently documented here. Therefore, the results below are treated as descriptive group differences. If the groups were randomized in the original experiment, the same differences could be interpreted more strongly as estimated treatment effects.

In [ ]:
coupon = coupon_raw.copy()

coupon_summary = coupon.groupby(
    ['pilot_group', 'coupon_amount']
).agg(
    customers=('user_id', 'count'),
    avg_30d_spend=('campaign_spend_30d', 'mean'),
    median_30d_spend=('campaign_spend_30d', 'median'),
    std_30d_spend=('campaign_spend_30d', 'std')
).reset_index().sort_values('coupon_amount')

control_mean = coupon_summary.loc[
    coupon_summary['coupon_amount'] == 0,
    'avg_30d_spend'
].iloc[0]

coupon_summary['uplift_vs_control'] = (
    coupon_summary['avg_30d_spend'] - control_mean
)

coupon_summary['uplift_pct_vs_control'] = (
    coupon_summary['uplift_vs_control']
    / control_mean
    * 100
)

coupon_summary['incremental_spend_per_coupon_value'] = np.where(
    coupon_summary['coupon_amount'] > 0,
    coupon_summary['uplift_vs_control']
    / coupon_summary['coupon_amount'],
    np.nan
)

display(coupon_summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=coupon_summary,
    x='coupon_amount',
    y='avg_30d_spend',
    ax=axes[0]
)
axes[0].set_title('Average 30-Day Spend by Coupon Amount')
axes[0].set_xlabel('Coupon Amount')
axes[0].set_ylabel('Average 30-Day Spend')

coupon_effect = coupon_summary[
    coupon_summary['coupon_amount'] > 0
]

sns.barplot(
    data=coupon_effect,
    x='coupon_amount',
    y='incremental_spend_per_coupon_value',
    ax=axes[1]
)
axes[1].set_title('Incremental Spend per Unit of Coupon Value')
axes[1].set_xlabel('Coupon Amount')
axes[1].set_ylabel('Incremental Spend / Coupon Value')

plt.tight_layout()
# Save high-resolution figure to Google Drive
image_path = f'{IMAGE_DIR}/coupon_analysis.png'
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Saved: {image_path}')

plt.show()

In [ ]:
# Optional: check whether each pilot group contains the same number of customers
display(
    coupon.groupby('pilot_group')['user_id']
    .nunique()
    .sort_values(ascending=False)
)

### Saved Portfolio Figures

After **Runtime → Run all**, the main figures are saved automatically to:

`MyDrive/Cafe_Customer_Analytics/images/`

These PNG files are suitable for direct insertion into the portfolio PPT/PDF.


In [ ]:
# Check saved portfolio images
expected_images = [
    'monthly_decomposition.png',
    'monthly_activity_trend.png',
    'pareto_curve.png',
    'category_analysis.png',
    'time_analysis.png',
    'customer_behavior_profile.png',
    'cluster_diagnostics.png',
    'pca_segments.png',
    'churn_distribution.png',
    'model_cross_validation.png',
    'confusion_matrix.png',
    'roc_pr_curves.png',
    'feature_importance.png',
    'risk_map.png',
    'coupon_analysis.png',
]

print("SAVED PORTFOLIO FIGURES")
print("-" * 60)

for filename in expected_images:
    path = f'{IMAGE_DIR}/{filename}'
    status = "✓" if os.path.exists(path) else "✗"
    print(f"{status} {path}")


### Coupon Interpretation

Two questions should be separated:

1. **Which coupon group has the highest observed 30-day spending?**
2. **Which coupon amount appears most efficient relative to its face value?**

The largest coupon can produce the highest spending while still producing diminishing incremental spending per unit of coupon value.

A real profitability decision would additionally require:
- redemption data
- product margin
- campaign delivery cost
- randomization confirmation

## 15. Recommended Actions

The analysis supports a targeted rather than one-size-fits-all approach.

### A. Retention
Prioritize customers with both:
- high churn probability
- high value-at-risk score

### B. Customer Segmentation
Use behavioral clusters to distinguish customers with different visit and spending patterns instead of treating all customers as average.

### C. Product Management
Track both:
- current revenue contribution
- direction of category demand

A large category is not automatically a growth category.

### D. Coupon Testing
Use the coupon pilot as evidence for a second-stage experiment rather than assuming that the largest discount is automatically the best economic choice.

### E. Measurement
For any future retention campaign, create:
- treatment and control groups
- pre-defined success metrics
- a fixed evaluation window

Recommended metrics:
- return rate
- incremental revenue
- margin after discount
- cost per retained customer

## 16. Limitations & Next Steps

### Limitations
1. The dataset is course-provided educational data rather than audited production data.
2. Customer churn is defined operationally as no purchase during October–December; another business may require a different churn window.
3. Coupon group randomization is not independently verified in this notebook.
4. The dataset contains transaction value but not full product cost, so campaign profitability cannot be calculated directly.
5. The value-at-risk score is a prioritization heuristic, not a causal estimate of revenue saved.

### Next Steps
- Add product-level margin and cost data
- Validate the churn model on a later time period
- Calibrate churn probabilities
- Tune campaign thresholds based on contact cost
- Run randomized retention experiments
- Track model drift over time

## 17. Executive Summary

In [ ]:
# Generate a concise, data-driven executive summary from the completed analysis

revenue_change = (
    monthly.iloc[-1]['revenue']
    / monthly.iloc[0]['revenue']
    - 1
) * 100

order_change = (
    monthly.iloc[-1]['orders']
    / monthly.iloc[0]['orders']
    - 1
) * 100

customer_change = (
    monthly.iloc[-1]['active_customers']
    / monthly.iloc[0]['active_customers']
    - 1
) * 100

aov_change = (
    monthly.iloc[-1]['avg_order_value']
    / monthly.iloc[0]['avg_order_value']
    - 1
) * 100

selected_test = test_results[
    test_results['Model'] == best_model_name
].iloc[0]

print("EXECUTIVE SUMMARY")
print("-" * 70)
print(f"1. 2025 Jan→Dec revenue change: {revenue_change:.1f}%")
print(f"2. 2025 Jan→Dec order-volume change: {order_change:.1f}%")
print(f"3. 2025 Jan→Dec active-customer change: {customer_change:.1f}%")
print(f"4. 2025 Jan→Dec average-order-value change: {aov_change:.1f}%")
print(f"5. Top 20% customer revenue share: {top_20_share:.1f}%")
print(f"6. Observed churn rate: {customer_features['churn'].mean():.1%}")
print(f"7. Selected churn model: {best_model_name}")
print(f"8. Held-out ROC-AUC: {selected_test['ROC_AUC']:.3f}")
print(f"9. Held-out Recall: {selected_test['Recall']:.3f}")
print(f"10. Priority retention list size: {len(priority_customers):,}")

## 18. Export Portfolio Outputs

The final cell saves the most useful tables to Google Drive so they can later be used in:
- the GitHub repository
- the portfolio PDF / PowerPoint
- additional analysis

In [ ]:
# Save portfolio-ready outputs
monthly.to_csv(
    f'{OUTPUT_DIR}/monthly_business_summary.csv',
    index=False
)

category_summary.reset_index().to_csv(
    f'{OUTPUT_DIR}/category_summary.csv',
    index=False
)

cluster_summary.reset_index().to_csv(
    f'{OUTPUT_DIR}/customer_cluster_summary.csv',
    index=False
)

cv_results.to_csv(
    f'{OUTPUT_DIR}/model_cross_validation.csv',
    index=False
)

test_results.to_csv(
    f'{OUTPUT_DIR}/model_test_results.csv',
    index=False
)

importance.to_csv(
    f'{OUTPUT_DIR}/churn_feature_importance.csv',
    index=False
)

risk_ranking.to_csv(
    f'{OUTPUT_DIR}/customer_risk_ranking.csv',
    index=False
)

coupon_summary.to_csv(
    f'{OUTPUT_DIR}/coupon_pilot_summary.csv',
    index=False
)

print("Portfolio outputs saved to:")
print(OUTPUT_DIR)

---

# Final Project Statement

This project demonstrates an end-to-end analytics workflow:

**raw transactions → data audit → business analysis → customer segmentation → churn prediction → interpretable risk ranking → marketing experiment analysis**

The most important lesson is that predictive accuracy alone is not the end goal.  
A useful analytics project must connect model outputs to a clear business decision while being transparent about assumptions and limitations.